# quantlab end-to-end demo

A single-pass walk-through of the full pipeline. The notebook deliberately uses a deterministic synthetic price panel so it runs offline; replace `synthetic_panel()` with `CSVCache(YFinanceSource()).fetch(...)` to use real Yahoo Finance data.

Pipeline stages:

1. **Data** — fetch / load a multi-ticker price panel.
2. **SQL** — in-memory SQLite analytical queries (window functions).
3. **Streaming** — online vol estimator and running median per ticker.
4. **Backtest** — parallel 12-1 momentum backtest across tickers.
5. **Forecast** — scikit-learn pipeline with TimeSeriesSplit cross-validation.
6. **Risk** — Monte Carlo VaR/CVaR (parallel) and historical simulation.
7. **DP baseline** — perfect-foresight upper bound for comparison.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
from generate_reports import synthetic_panel

panel = synthetic_panel(n_days=1500, seed=42)
panel.head()

## 1. SQL window-function queries

In [ ]:
from quantlab.data.sql_layer import SQLAnalytics

with SQLAnalytics(panel.assign(close=panel['close'].astype(float))) as sql:
    avg_vol = sql.rolling_avg_volume(window=20)
    momentum = sql.momentum_signal(lookback=252)
    rank = sql.cross_sectional_rank()

print(avg_vol.head())
print(momentum.dropna().head())
print(rank.head())

## 2. Streaming statistics per ticker

In [ ]:
from quantlab.compute.rolling import log_returns
from quantlab.streaming.median import RunningMedian
from quantlab.streaming.welford import OnlineMoments

results = {}
for ticker, sub in panel.groupby('ticker'):
    rets = log_returns(sub.set_index('date')['close'])
    moments = OnlineMoments()
    median = RunningMedian()
    moments.update_many(rets.values)
    median.add_many(rets.values)
    results[ticker] = {'mean': moments.mean, 'std': moments.std, 'median': median.median}

import pandas as pd
pd.DataFrame(results).T

## 3. Parallel momentum backtest

In [ ]:
from quantlab.compute.backtest import momentum_strategy, run_backtest

result = run_backtest(panel, strategy=lambda d: momentum_strategy(d, lookback=252, skip=21),
                       n_workers=1)
result.metrics_table()

In [ ]:
from quantlab.viz.returns import plot_cumulative_returns
from quantlab.viz.drawdown import plot_drawdown

plot_cumulative_returns(result.portfolio_equity.rename('portfolio'),
                         title='Equal-weight momentum portfolio');
plot_drawdown(result.portfolio_equity, title='Portfolio drawdown');

## 4. Forecasting (single ticker, classification)

In [ ]:
from quantlab.models.evaluation import classification_metrics
from quantlab.models.features import build_features
from quantlab.models.forecaster import ReturnForecaster

apple = panel[panel['ticker'] == 'AAPL'].set_index('date').sort_index()
feats = build_features(apple)
fc = ReturnForecaster(task='classification', n_splits=5)
fit = fc.fit(feats)
print('CV mean AUC:', round(fit.cv_mean, 4))

proba = fc.predict_proba(feats)
classification_metrics(feats['y_next_dir'], proba)

## 5. Risk: Monte Carlo VaR vs historical simulation

In [ ]:
from quantlab.compute.montecarlo import historical_simulation, monte_carlo_var

rets = log_returns(apple['close']).dropna()
mc = monte_carlo_var(mu=float(rets.mean()), sigma=float(rets.std(ddof=1)),
                     horizon_days=10, n_paths=20_000, confidence=0.99,
                     n_workers=1, seed=0)
hist = historical_simulation(rets.values, confidence=0.99)
print(f'Monte Carlo: VaR={mc.var:.4%}  CVaR={mc.cvar:.4%}')
print(f'Historical:  VaR={hist.var:.4%}  CVaR={hist.cvar:.4%}')

## 6. DP baseline: perfect-foresight upper bound

In [ ]:
from quantlab.compute.optimal_execution import max_profit_with_fee, max_profit_with_k_trades

prices = apple['close'].tolist()
print('Per-share max profit (fee=0.001):', round(max_profit_with_fee(prices, fee=0.001), 2))
print('At most 5 round-trips:           ', round(max_profit_with_k_trades(prices, k=5), 2))

## 7. Markowitz portfolio

In [ ]:
import numpy as np
from quantlab.portfolio.markowitz import mean_variance_optimal

wide = pd.DataFrame({t: log_returns(s.set_index('date')['close'])
                       for t, s in panel.groupby('ticker')}).dropna()
mu = wide.mean().values * 252
cov = wide.cov().values * 252
res = mean_variance_optimal(mu, cov, risk_aversion=2.0)
pd.Series(res.weights, index=wide.columns).round(4)